# Imports and Setup


In [ ]:
# Install the augmentation dependency and import all libraries used in the notebook.
!pip install -q albumentations

import os
import random
import xml.etree.ElementTree as ET
from collections import Counter

import albumentations as A
import cv2
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision.models import resnet34
from tqdm.auto import tqdm


# Reproducibility


In [ ]:
# Set random seeds so data splits, sampling, and model initialization are reproducible.

# Set a fixed random seed for reproducible results across Python, NumPy, and PyTorch.

# ==========================================================
# Reproducibility / Global Seed
# ==========================================================

SEED = 42

# Python
random.seed(SEED)

# Numpy
np.random.seed(SEED)

# Hash seed
os.environ["PYTHONHASHSEED"] = str(SEED)

# PyTorch
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# CuDNN
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Seed set to {SEED}")

# Exploratory Data Analysis (Already in a separate notebook)


In [ ]:
# Define dataset paths and class labels, then preview the dataset folder structure.

DATASET_PATH = "/kaggle/input/datasets/alimaged10/pcb-dataset/PCB-DATASET-master"
CLASSES = [
    "Missing_hole",
    "Mouse_bite",
    "Open_circuit",
    "Short",
    "Spur",
    "Spurious_copper"
]

# Keep lowercase aliases for the EDA cells that use the original names.
dataset_path = DATASET_PATH
classes = CLASSES

for root, dirs, files in os.walk(dataset_path):
    level = root.replace(dataset_path, '').count(os.sep)
    indent = ' ' * 4 * level
    print(f"{indent}{os.path.basename(root)}/")
    
    if level < 2:  # prevent huge output
        subindent = ' ' * 4 * (level + 1)
        for f in files[:10]:
            print(f"{subindent}{f}")


# Model: ResNet-Backbone PCB Detector


## Phase 1: Build Annotation Records and Dataset Splits


In [ ]:
# Build one annotation record per image and split records into train, validation, and test sets.

# ==========================================================
# Phase 1 - Data Preparation
# ==========================================================


CLASS_TO_ID = {
    cls_name: idx
    for idx, cls_name in enumerate(CLASSES)
}

ID_TO_CLASS = {
    idx: cls_name
    for idx, cls_name in enumerate(CLASSES)
}

# ----------------------------------------------------------
# Build Records
# ----------------------------------------------------------

records = []

for cls_name in CLASSES:

    img_dir = os.path.join(
        DATASET_PATH,
        "images",
        cls_name
    )

    ann_dir = os.path.join(
        DATASET_PATH,
        "Annotations",
        cls_name
    )

    for xml_file in os.listdir(ann_dir):

        if not xml_file.endswith(".xml"):
            continue

        xml_path = os.path.join(
            ann_dir,
            xml_file
        )

        image_name = xml_file.replace(
            ".xml",
            ".jpg"
        )

        image_path = os.path.join(
            img_dir,
            image_name
        )

        if not os.path.exists(image_path):
            continue

        tree = ET.parse(xml_path)
        root = tree.getroot()

        boxes = []
        labels = []

        for obj in root.findall("object"):

            bbox = obj.find("bndbox")

            xmin = int(bbox.find("xmin").text)
            ymin = int(bbox.find("ymin").text)
            xmax = int(bbox.find("xmax").text)
            ymax = int(bbox.find("ymax").text)

            boxes.append(
                [xmin, ymin, xmax, ymax]
            )

            labels.append(
                CLASS_TO_ID[cls_name]
            )

        record = {
            "image_path": image_path,
            "boxes": boxes,
            "labels": labels,
            "class_name": cls_name
        }

        records.append(record)

# ----------------------------------------------------------
# Basic Verification
# ----------------------------------------------------------

print("=" * 60)
print("TOTAL IMAGES")
print("=" * 60)

print(len(records))

print("\nExample Record:\n")

print(records[0])

# ----------------------------------------------------------
# Split
# ----------------------------------------------------------
#
# 70% Train
# 20% Validation
# 10% Test
#
# First:
#   Train = 70%
#   Temp  = 30%
#
# Then:
#   Val = 20%
#   Test = 10%
#
# Inside temp:
#   Val = 2/3
#   Test = 1/3
#
# ----------------------------------------------------------

# ----------------------------------------------------------
# Stratification Labels
# ----------------------------------------------------------

stratify_labels = [
    record["class_name"]
    for record in records
]

# ----------------------------------------------------------
# First Split
# ----------------------------------------------------------

train_records, temp_records = train_test_split(
    records,
    test_size=0.30,
    random_state=42,
    shuffle=True,
    stratify=stratify_labels
)

# ----------------------------------------------------------
# Labels For Temp Split
# ----------------------------------------------------------

temp_labels = [
    record["class_name"]
    for record in temp_records
]

# ----------------------------------------------------------
# Second Split
# ----------------------------------------------------------

val_records, test_records = train_test_split(
    temp_records,
    test_size=(1/3),
    random_state=42,
    shuffle=True,
    stratify=temp_labels
)

# ----------------------------------------------------------
# Verify Split Sizes
# ----------------------------------------------------------

print("\n" + "=" * 60)
print("SPLIT SIZES")
print("=" * 60)

print(f"Train: {len(train_records)}")
print(f"Val  : {len(val_records)}")
print(f"Test : {len(test_records)}")

# ----------------------------------------------------------
# Class Distribution
# ----------------------------------------------------------

def count_classes(record_list):

    counter = Counter()

    for rec in record_list:

        for label in rec["labels"]:
            counter[label] += 1

    return counter

train_counts = count_classes(train_records)
val_counts = count_classes(val_records)
test_counts = count_classes(test_records)

print("\n" + "=" * 60)
print("TRAIN CLASS COUNTS")
print("=" * 60)

for idx, count in sorted(train_counts.items()):
    print(
        f"{ID_TO_CLASS[idx]:<20} {count}"
    )

print("\n" + "=" * 60)
print("VAL CLASS COUNTS")
print("=" * 60)

for idx, count in sorted(val_counts.items()):
    print(
        f"{ID_TO_CLASS[idx]:<20} {count}"
    )

print("\n" + "=" * 60)
print("TEST CLASS COUNTS")
print("=" * 60)

for idx, count in sorted(test_counts.items()):
    print(
        f"{ID_TO_CLASS[idx]:<20} {count}"
    )

# ----------------------------------------------------------
# Save Split Lists
# ----------------------------------------------------------

print("\n")
print("Data preparation completed.")

In [ ]:
# Count total labeled defects across the full dataset.

total_counter = Counter()

for rec in records:
    for label in rec["labels"]:
        total_counter[label] += 1

print("\nTOTAL DATASET COUNTS")

for idx, count in sorted(total_counter.items()):
    print(
        f"{ID_TO_CLASS[idx]:<20} {count}"
    )

In [ ]:
# Count how many image records from each class are present in each split.

def count_images_per_class(records):

    counter = Counter()

    for rec in records:
        counter[rec["class_name"]] += 1

    return counter

print("\nTRAIN IMAGE COUNTS")
print(count_images_per_class(train_records))

print("\nVAL IMAGE COUNTS")
print(count_images_per_class(val_records))

print("\nTEST IMAGE COUNTS")
print(count_images_per_class(test_records))

## Phase 2: Define Image Size, Augmentations, and Dataset Objects


In [ ]:
# Set the shared image size and batch size used by the data pipeline.

IMAGE_SIZE = 1024
BATCH_SIZE = 2

In [ ]:
# Define the training augmentation pipeline with flips, resizing, normalization, and tensor conversion.

train_transform = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),

        A.Resize(
            height=IMAGE_SIZE,
            width=IMAGE_SIZE
        ),

        A.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),

        ToTensorV2()
    ],
    bbox_params=A.BboxParams(
        format="pascal_voc",
        label_fields=["labels"]
    )
)

In [ ]:
# Define deterministic validation and test preprocessing.

val_test_transform = A.Compose(
    [
        A.Resize(
            height=IMAGE_SIZE,
            width=IMAGE_SIZE
        ),
        A.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
),

        ToTensorV2()
    ],
    bbox_params=A.BboxParams(
        format="pascal_voc",
        label_fields=["labels"]
    )
)

In [ ]:
# Implement the PyTorch dataset that loads images, boxes, and labels.

class PCBDataset(Dataset):

    def __init__(
        self,
        records,
        transform=None
    ):

        self.records = records
        self.transform = transform

    def __len__(self):

        return len(self.records)

    def __getitem__(self, idx):

        record = self.records[idx]

        image_path = record["image_path"]

        image = cv2.imread(image_path)

        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )

        boxes = record["boxes"]
        labels = record["labels"]

        if self.transform:

            transformed = self.transform(
                image=image,
                bboxes=boxes,
                labels=labels
            )

            image = transformed["image"]

            boxes = transformed["bboxes"]

            labels = transformed["labels"]

        boxes = torch.tensor(
            boxes,
            dtype=torch.float32
        )

        labels = torch.tensor(
            labels,
            dtype=torch.long
        )

        target = {
            "boxes": boxes,
            "labels": labels
        }

        return image, target

In [ ]:
# Create train, validation, and test dataset objects.

train_dataset = PCBDataset(
    train_records,
    transform=train_transform
)

val_dataset = PCBDataset(
    val_records,
    transform=val_test_transform
)

test_dataset = PCBDataset(
    test_records,
    transform=val_test_transform
)

In [ ]:
# Print the number of images in each dataset split.

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))

In [ ]:
# Inspect one transformed sample image and its target tensor shapes.

image, target = train_dataset[0]

print("Image Shape:", image.shape)

print()

print("Boxes Shape:")
print(target["boxes"].shape)

print()

print("Labels Shape:")
print(target["labels"].shape)

In [ ]:
# Visualize one transformed training sample with its bounding boxes.

# --------------------------------------------------
# Sample
# --------------------------------------------------

image, target = train_dataset[0]

# --------------------------------------------------
# De-normalize
# --------------------------------------------------

mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])

img = image.permute(1, 2, 0).cpu().numpy()

img = img * std + mean

img = np.clip(img, 0, 1)

# --------------------------------------------------
# Plot
# --------------------------------------------------

fig, ax = plt.subplots(figsize=(10,10))

ax.imshow(img)

for box, label in zip(
    target["boxes"],
    target["labels"]
):

    xmin, ymin, xmax, ymax = box.numpy()

    rect = patches.Rectangle(
        (xmin, ymin),
        xmax - xmin,
        ymax - ymin,
        linewidth=2,
        edgecolor="red",
        fill=False
    )

    ax.add_patch(rect)

    ax.text(
        xmin,
        ymin - 5,
        ID_TO_CLASS[label.item()],
        color="red",
        fontsize=8,
        bbox=dict(
            facecolor="white",
            alpha=0.7,
            edgecolor="none"
        )
    )

plt.axis("off")
plt.show()

## Phase 3: Build and Inspect the ResNet34 Backbone


In [ ]:
# Define a ResNet34 backbone that returns intermediate feature maps.

# ==========================================================
# Phase 3 - ResNet34 Backbone
# ==========================================================


class ResNet34Backbone(nn.Module):

    def __init__(self):

        super().__init__()

        model = resnet34(weights="DEFAULT")

        self.stem = nn.Sequential(
            model.conv1,
            model.bn1,
            model.relu,
            model.maxpool
        )

        self.layer1 = model.layer1
        self.layer2 = model.layer2
        self.layer3 = model.layer3
        self.layer4 = model.layer4

    def forward(self, x):

        x = self.stem(x)

        c1 = self.layer1(x)
        c2 = self.layer2(c1)
        c3 = self.layer3(c2)
        c4 = self.layer4(c3)

        return c1, c2, c3, c4

In [ ]:
# Select the compute device and instantiate the backbone.

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

backbone = ResNet34Backbone().to(device)

In [ ]:
# Run one sample through the backbone and inspect feature-map shapes.

image, target = train_dataset[0]

x = image.unsqueeze(0).to(device)

with torch.no_grad():
    
    c1, c2, c3, c4 = backbone(x)

print("C1:", c1.shape)
print("C2:", c2.shape)
print("C3:", c3.shape)
print("C4:", c4.shape)

In [ ]:
# Check the normalized tensor dtype and value range for one training image.

image, target = train_dataset[0]

print(image.dtype)
print(image.min())
print(image.max())

## Phase 4: Fuse Multi-Scale Backbone Features


In [ ]:
# Define the feature-fusion module that combines C3 and C4 backbone features.

# ==========================================================
# Phase 4 - Feature Fusion Module
# ==========================================================


class FeatureFusion(nn.Module):

    def __init__(self):

        super().__init__()

        # Convert C4 (512 channels)
        # to match C3 (256 channels)

        self.c4_reduce = nn.Conv2d(
            in_channels=512,
            out_channels=256,
            kernel_size=1
        )

        # Smooth fused output

        self.fusion_conv = nn.Sequential(

            nn.Conv2d(
                256,
                256,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(256),

            nn.ReLU(inplace=True)
        )

    def forward(self, c3, c4):

        # ------------------------------
        # Reduce channels
        # ------------------------------

        c4 = self.c4_reduce(c4)

        # ------------------------------
        # Upsample C4
        # ------------------------------

        c4 = F.interpolate(
            c4,
            size=c3.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

        # ------------------------------
        # Fusion
        # ------------------------------

        fused = c3 + c4

        # ------------------------------
        # Refinement
        # ------------------------------

        fused = self.fusion_conv(fused)

        return fused

In [ ]:
# Instantiate the feature-fusion module on the active device.

fusion = FeatureFusion().to(device)

In [ ]:
# Verify the fused feature-map shape on one sample image.

image, target = train_dataset[0]

x = image.unsqueeze(0).to(device)

with torch.no_grad():

    c1, c2, c3, c4 = backbone(x)

    fused = fusion(c3, c4)

print("C3:", c3.shape)
print("C4:", c4.shape)
print("Fused:", fused.shape)

In [ ]:
# Inspect basic statistics of the fused feature map.

print(fused.min())
print(fused.max())
print(fused.mean())

## Phase 5: Add the Detection Head


In [ ]:
# Define the detection head that predicts objectness, class logits, and box offsets.

# ==========================================================
# Phase 5 - Detection Head
# ==========================================================


class DetectionHead(nn.Module):

    def __init__(self, num_classes=6):

        super().__init__()

        # ------------------------------------------
        # Shared Feature Processing
        # ------------------------------------------

        self.shared = nn.Sequential(

            nn.Conv2d(
                in_channels=256,
                out_channels=256,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(256),

            nn.ReLU(inplace=True),

            nn.Conv2d(
                in_channels=256,
                out_channels=128,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(128),

            nn.ReLU(inplace=True)
        )

        # ------------------------------------------
        # Objectness Branch
        # ------------------------------------------

        self.objectness_head = nn.Conv2d(
            in_channels=128,
            out_channels=1,
            kernel_size=1
        )

        # ------------------------------------------
        # Classification Branch
        # ------------------------------------------

        self.class_head = nn.Conv2d(
            in_channels=128,
            out_channels=num_classes,
            kernel_size=1
        )

        # ------------------------------------------
        # Bounding Box Branch
        # ------------------------------------------

        self.box_head = nn.Conv2d(
            in_channels=128,
            out_channels=4,
            kernel_size=1
        )


    def forward(self, x):
    
        x = self.shared(x)
    
        objectness = self.objectness_head(x)
    
        class_logits = self.class_head(x)
    
        # ------------------------------------------
        # Bounding Boxes
        #
        # tx, ty, w, h
        # constrained to [0,1]
        # ------------------------------------------
    
        bbox_regression = torch.sigmoid(
            self.box_head(x)
        )
    
        return (
            objectness,
            class_logits,
            bbox_regression
        )

In [ ]:
# Instantiate the detection head for the PCB defect classes.

detector_head = DetectionHead(
    num_classes=len(CLASSES)
).to(device)

In [ ]:
# Run the backbone, fusion module, and detection head on one sample.

image, target = train_dataset[0]

x = image.unsqueeze(0).to(device)

with torch.no_grad():

    c1, c2, c3, c4 = backbone(x)

    fused = fusion(c3, c4)

    objectness, classes, boxes = detector_head(fused)

print("Objectness:", objectness.shape)
print("Classes   :", classes.shape)
print("Boxes     :", boxes.shape)

In [ ]:
# Inspect the raw prediction ranges before training.

print()

print("Objectness Range:")
print(objectness.min().item())
print(objectness.max().item())

print()

print("Class Range:")
print(classes.min().item())
print(classes.max().item())

print()

print("Box Range:")
print(boxes.min().item())
print(boxes.max().item())

## Phase 6: Encode Bounding Boxes as Grid Targets


In [ ]:
# Define the grid layout used to encode detection targets.

GRID_SIZE = 64
CELL_SIZE = IMAGE_SIZE / GRID_SIZE
NUM_CLASSES = len(CLASSES)

print("Cell Size:", CELL_SIZE)


In [ ]:
# Encode ground-truth boxes and labels into objectness, class, and box target maps.

def encode_targets(
    boxes,
    labels,
    image_size=1024,
    grid_size=64
):

    objectness = torch.zeros(
        (grid_size, grid_size),
        dtype=torch.float32
    )

    class_target = torch.full(
        (grid_size, grid_size),
        -1,
        dtype=torch.long
    )

    bbox_target = torch.zeros(
        (4, grid_size, grid_size),
        dtype=torch.float32
    )

    cell_size = image_size / grid_size

    for box, label in zip(boxes, labels):

        xmin, ymin, xmax, ymax = box.tolist()

        # ------------------------------------------
        # Box Center
        # ------------------------------------------

        cx = (xmin + xmax) / 2
        cy = (ymin + ymax) / 2

        # ------------------------------------------
        # Box Size
        # ------------------------------------------

        w = xmax - xmin
        h = ymax - ymin

        # ------------------------------------------
        # Responsible Cell
        # ------------------------------------------

        grid_x = int(cx / cell_size)
        grid_y = int(cy / cell_size)

        grid_x = min(grid_x, grid_size - 1)
        grid_y = min(grid_y, grid_size - 1)

        # ------------------------------------------
        # Relative Offset Inside Cell
        #
        # tx, ty in [0,1]
        # ------------------------------------------

        tx = (cx / cell_size) - grid_x
        ty = (cy / cell_size) - grid_y

        # ------------------------------------------
        # Objectness
        # ------------------------------------------

        objectness[grid_y, grid_x] = 1.0

        # ------------------------------------------
        # Class
        # ------------------------------------------

        class_target[grid_y, grid_x] = label

        # ------------------------------------------
        # BBox Targets
        #
        # [tx, ty, w, h]
        # ------------------------------------------

        bbox_target[0, grid_y, grid_x] = tx
        bbox_target[1, grid_y, grid_x] = ty

        bbox_target[2, grid_y, grid_x] = (
            w / image_size
        )

        bbox_target[3, grid_y, grid_x] = (
            h / image_size
        )

    return (
        objectness,
        class_target,
        bbox_target
    )

In [ ]:
# Encode one training sample and inspect the target-map shapes.

image, target = train_dataset[0]

objectness_target, class_target, bbox_target = encode_targets(
    target["boxes"],
    target["labels"]
)

print("Objectness:", objectness_target.shape)
print("Class:", class_target.shape)
print("BBox:", bbox_target.shape)

print()

print("Positive Cells:")
print(objectness_target.sum())

In [ ]:
# Locate grid cells that contain ground-truth objects.

positive_cells = torch.where(
    objectness_target == 1
)

print("Y:", positive_cells[0])
print("X:", positive_cells[1])

In [ ]:
# Print the positive grid-cell coordinates.

for y, x in zip(
    positive_cells[0],
    positive_cells[1]
):
    print(
        f"Cell ({x.item()}, {y.item()})"
    )

In [ ]:
# Visualize encoded target cells against the original bounding boxes.

# ------------------------------------------
# Sample
# ------------------------------------------

image, target = train_dataset[0]

objectness_target, class_target, bbox_target = encode_targets(
    target["boxes"],
    target["labels"]
)

# ------------------------------------------
# De-normalize image
# ------------------------------------------

mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])

img = image.permute(1, 2, 0).cpu().numpy()
img = img * std + mean
img = np.clip(img, 0, 1)

# ------------------------------------------
# Plot
# ------------------------------------------

fig, ax = plt.subplots(figsize=(12,12))

ax.imshow(img)

# ------------------------------------------
# Draw Bounding Boxes
# ------------------------------------------

for box in target["boxes"]:

    xmin, ymin, xmax, ymax = box.numpy()

    rect = patches.Rectangle(
        (xmin, ymin),
        xmax - xmin,
        ymax - ymin,
        linewidth=2,
        edgecolor="red",
        fill=False
    )

    ax.add_patch(rect)

# ------------------------------------------
# Draw Positive Cells
# ------------------------------------------

positive_cells = torch.where(
    objectness_target == 1
)

cell_size = 1024 / 64

for y, x in zip(
    positive_cells[0],
    positive_cells[1]
):

    x = x.item()
    y = y.item()

    rect = patches.Rectangle(
        (x * cell_size, y * cell_size),
        cell_size,
        cell_size,
        linewidth=2,
        edgecolor="lime",
        fill=False
    )

    ax.add_patch(rect)

# ------------------------------------------
# Grid
# ------------------------------------------

for i in range(65):

    ax.axhline(
        i * cell_size,
        color="yellow",
        linewidth=0.2,
        alpha=1
    )

    ax.axvline(
        i * cell_size,
        color="yellow",
        linewidth=0.2,
        alpha=1
    )

plt.title(
    "Red = Ground Truth Boxes | Green = Assigned Cells"
)

plt.axis("off")
plt.show()

In [ ]:
# Display one normalized sample after reversing normalization for visualization.

image, target = train_dataset[0]

img = image.permute(1,2,0).cpu().numpy()

mean = np.array([0.485,0.456,0.406])
std  = np.array([0.229,0.224,0.225])

img = img * std + mean
img = np.clip(img,0,1)

fig, ax = plt.subplots(figsize=(12,12))
ax.imshow(img)

for box in target["boxes"]:

    xmin, ymin, xmax, ymax = box.numpy()

    cx = (xmin + xmax) / 2
    cy = (ymin + ymax) / 2

    ax.scatter(
        cx,
        cy,
        s=80,
        c="cyan"
    )

    rect = patches.Rectangle(
        (xmin, ymin),
        xmax - xmin,
        ymax - ymin,
        linewidth=2,
        edgecolor="red",
        fill=False
    )

    ax.add_patch(rect)

plt.title(
    "Cyan = Box Centers"
)

plt.axis("off")
plt.show()

In [ ]:
# Show one random training example per class with encoded target information.

# ------------------------------------------
# One random sample per class
# ------------------------------------------

selected_records = []

for cls in CLASSES:

    class_indices = [
        i
        for i, rec in enumerate(train_records)
        if rec["class_name"] == cls
    ]

    selected_records.append(
        random.choice(class_indices)
    )

# ------------------------------------------
# Plot
# ------------------------------------------

fig, axes = plt.subplots(
    2,
    3,
    figsize=(18,12)
)

axes = axes.flatten()

for ax, idx in zip(axes, selected_records):

    image, target = train_dataset[idx]

    objectness_target, class_target, bbox_target = encode_targets(
        target["boxes"],
        target["labels"]
    )

    # ----------------------------
    # De-normalize
    # ----------------------------

    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])

    img = image.permute(1,2,0).cpu().numpy()

    img = img * std + mean
    img = np.clip(img, 0, 1)

    ax.imshow(img)

    # ----------------------------
    # GT Boxes
    # ----------------------------

    for box in target["boxes"]:

        xmin, ymin, xmax, ymax = box.numpy()

        rect = patches.Rectangle(
            (xmin, ymin),
            xmax - xmin,
            ymax - ymin,
            linewidth=2,
            edgecolor="red",
            fill=False
        )

        ax.add_patch(rect)

    # ----------------------------
    # Positive Cells
    # ----------------------------

    positive_cells = torch.where(
        objectness_target == 1
    )

    cell_size = 1024 / 64

    for y, x in zip(
        positive_cells[0],
        positive_cells[1]
    ):

        x = x.item()
        y = y.item()

        rect = patches.Rectangle(
            (x * cell_size, y * cell_size),
            cell_size,
            cell_size,
            linewidth=2,
            edgecolor="lime",
            fill=False
        )

        ax.add_patch(rect)

    # ----------------------------
    # Title
    # ----------------------------

    label = target["labels"][0].item()

    ax.set_title(
        ID_TO_CLASS[label]
    )

    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Visualize grid alignment and encoded boxes for random samples.

mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])

cell_size = IMAGE_SIZE / GRID_SIZE

# ==========================================================
# Show One Random Example Per Class
# ==========================================================

for cls in CLASSES:

    # ------------------------------------------------------
    # Pick Random Sample From This Class
    # ------------------------------------------------------

    indices = [
        i
        for i, rec in enumerate(train_records)
        if rec["class_name"] == cls
    ]

    idx = random.choice(indices)

    image, target = train_dataset[idx]

    objectness_target, class_target, bbox_target = encode_targets(
        target["boxes"],
        target["labels"]
    )

    # ------------------------------------------------------
    # De-normalize
    # ------------------------------------------------------

    img = image.permute(1, 2, 0).cpu().numpy()

    img = img * std + mean
    img = np.clip(img, 0, 1)

    # ------------------------------------------------------
    # Plot
    # ------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(12, 12)
    )

    ax.imshow(img)

    # ------------------------------------------------------
    # Draw GT Boxes + Centers
    # ------------------------------------------------------

    for box in target["boxes"]:

        xmin, ymin, xmax, ymax = box.numpy()

        # Ground Truth Box

        rect = patches.Rectangle(
            (xmin, ymin),
            xmax - xmin,
            ymax - ymin,
            linewidth=2,
            edgecolor="red",
            fill=False
        )

        ax.add_patch(rect)

        # Center

        cx = (xmin + xmax) / 2
        cy = (ymin + ymax) / 2

        ax.scatter(
            cx,
            cy,
            color="cyan",
            s=50
        )

    # ------------------------------------------------------
    # Assigned Cells
    # ------------------------------------------------------

    positive_cells = torch.where(
        objectness_target == 1
    )

    for y, x in zip(
        positive_cells[0],
        positive_cells[1]
    ):

        x = x.item()
        y = y.item()

        rect = patches.Rectangle(
            (
                x * cell_size,
                y * cell_size
            ),
            cell_size,
            cell_size,
            linewidth=2,
            edgecolor="lime",
            fill=False
        )

        ax.add_patch(rect)

    # ------------------------------------------------------
    # Grid
    # ------------------------------------------------------

    for i in range(GRID_SIZE + 1):

        ax.axhline(
            i * cell_size,
            color="yellow",
            linewidth=0.2,
            alpha=1
        )

        ax.axvline(
            i * cell_size,
            color="yellow",
            linewidth=0.2,
            alpha=1
        )

    ax.set_title(
        f"{cls}\nRed=GT Box | Cyan=Center | Green=Assigned Cell",
        fontsize=14
    )

    ax.axis("off")

    plt.show()

## Phase 7: Define and Test the Detection Loss


In [ ]:
# Define the combined objectness, classification, and box regression loss.

# ==========================================================
# Phase 7 - Detection Loss
# ==========================================================


class DetectionLoss(nn.Module):

    def __init__(self):

        super().__init__()

        self.register_buffer(
            "pos_weight",
            torch.tensor([50.0])
        )
        
        self.objectness_loss = nn.BCEWithLogitsLoss(
            pos_weight=self.pos_weight
        )

        self.classification_loss = nn.CrossEntropyLoss(
            reduction="none"
        )

        self.bbox_loss = nn.SmoothL1Loss(
            reduction="none"
        )

    def forward(

        self,

        objectness_pred,
        class_pred,
        bbox_pred,

        objectness_target,
        class_target,
        bbox_target
    ):

        # ==========================================
        # Objectness
        # ==========================================

        obj_loss = self.objectness_loss(
            objectness_pred.squeeze(1),
            objectness_target
        )

        # ==========================================
        # Positive Cell Mask
        # ==========================================

        positive_mask = (
            objectness_target == 1
        )

        num_positive = positive_mask.sum()

        # ==========================================
        # Classification
        # ==========================================

        if num_positive > 0:

            class_loss_all = self.classification_loss(
                class_pred,
                class_target.clamp(min=0)
            )

            class_loss = class_loss_all[
                positive_mask
            ].mean()

        else:

            class_loss = torch.tensor(
                0.0,
                device=objectness_pred.device
            )

        # ==========================================
        # Bounding Box
        # ==========================================

        if num_positive > 0:

            bbox_loss_all = self.bbox_loss(
                bbox_pred,
                bbox_target
            )

            bbox_mask = (
                positive_mask
                .unsqueeze(1)
                .expand_as(bbox_pred)
            )

            bbox_loss = bbox_loss_all[
                bbox_mask
            ].mean()

        else:

            bbox_loss = torch.tensor(
                0.0,
                device=objectness_pred.device
            )

        # ==========================================
        # Total
        # ==========================================

        total_loss = (
            obj_loss
            + class_loss
            + bbox_loss
        )

        return {

            "total_loss": total_loss,

            "objectness_loss": obj_loss,

            "classification_loss": class_loss,

            "bbox_loss": bbox_loss
        }

In [ ]:
# Instantiate the detection loss.

criterion = DetectionLoss().to(device)

In [ ]:
# Check that the positive-class weight tensor is on the expected device.

print(criterion.pos_weight.device)

In [ ]:
# Build target maps for one sample to test the loss inputs.

image, target = train_dataset[0]

objectness_target, class_target, bbox_target = encode_targets(
    target["boxes"],
    target["labels"]
)

print(objectness_target.shape)
print(class_target.shape)
print(bbox_target.shape)

In [ ]:
# Generate predictions for one sample without updating model weights.

x = image.unsqueeze(0).to(device)

with torch.no_grad():

    c1, c2, c3, c4 = backbone(x)

    fused = fusion(c3, c4)

    objectness_pred, class_pred, bbox_pred = detector_head(
        fused
    )

In [ ]:
# Add a batch dimension and move target maps to the active device.

objectness_target = objectness_target.unsqueeze(0).to(device)

class_target = class_target.unsqueeze(0).to(device)

bbox_target = bbox_target.unsqueeze(0).to(device)

In [ ]:
# Compute the loss components for one sample.

losses = criterion(

    objectness_pred,
    class_pred,
    bbox_pred,

    objectness_target,
    class_target,
    bbox_target
)

In [ ]:
# Print each loss component for inspection.

for name, value in losses.items():

    print(
        f"{name}: {value.item():.6f}"
    )

## Phase 8: Assemble the Full Detector and DataLoaders


In [ ]:
# Combine the backbone, fusion module, and detection head into one detector.

# ==========================================================
# Phase 8.1 - Full Detector
# ==========================================================


class PCBDetector(nn.Module):

    def __init__(
        self,
        num_classes=6
    ):

        super().__init__()

        self.backbone = ResNet34Backbone()

        self.fusion = FeatureFusion()

        self.head = DetectionHead(
            num_classes=num_classes
        )

    def forward(self, x):

        c1, c2, c3, c4 = self.backbone(x)

        fused = self.fusion(
            c3,
            c4
        )

        objectness_pred, class_pred, bbox_pred = self.head(
            fused
        )

        return (
            objectness_pred,
            class_pred,
            bbox_pred
        )

In [ ]:
# Instantiate the full detector and report parameter counts.

model = PCBDetector(
    num_classes=len(CLASSES)
).to(device)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    f"Total Parameters: {total_params:,}"
)

print(
    f"Trainable Parameters: {trainable_params:,}"
)

In [ ]:
# Verify the full detector output shapes on one sample image.

image, target = train_dataset[0]

x = image.unsqueeze(0).to(device)

with torch.no_grad():

    objectness_pred, class_pred, bbox_pred = model(x)

print(
    "Objectness:",
    objectness_pred.shape
)

print(
    "Classes:",
    class_pred.shape
)

print(
    "Boxes:",
    bbox_pred.shape
)

In [ ]:
# Define the custom collate function for variable numbers of boxes per image.

def collate_fn(batch):

    images = []
    targets = []

    for image, target in batch:

        images.append(image)

        targets.append(target)

    images = torch.stack(images)

    return images, targets

In [ ]:
# Build train, validation, and test DataLoaders.

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn
)

In [ ]:
# Inspect one batch from the training DataLoader.

images, targets = next(
    iter(train_loader)
)

print(
    "Images:",
    images.shape
)

print(
    "Targets:",
    len(targets)
)

print(
    "Boxes shape:",
    targets[0]["boxes"].shape
)

print(
    "Labels shape:",
    targets[0]["labels"].shape
)

In [ ]:
# Convert a batch of targets into stacked target maps.

images, targets = next(
    iter(train_loader)
)

objectness_targets = []
class_targets = []
bbox_targets = []

for target in targets:

    obj_tgt, cls_tgt, box_tgt = encode_targets(
        target["boxes"],
        target["labels"]
    )

    objectness_targets.append(obj_tgt)
    class_targets.append(cls_tgt)
    bbox_targets.append(box_tgt)

objectness_targets = torch.stack(
    objectness_targets
)

class_targets = torch.stack(
    class_targets
)

bbox_targets = torch.stack(
    bbox_targets
)

print(
    objectness_targets.shape
)

print(
    class_targets.shape
)

print(
    bbox_targets.shape
)

## Phase 9: Train the Detector with Validation and Early Stopping


In [ ]:
# Configure the optimizer and learning-rate scheduler.

# ==========================================================
# Phase 9.1 - Optimizer + Scheduler
# ==========================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3
)

In [ ]:
# Define one training epoch over the training DataLoader.

def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device
):

    model.train()

    running_loss = 0.0

    progress_bar = tqdm(
        loader,
        desc="Training",
        leave=False
    )

    for images, targets in progress_bar:

        images = images.to(device)

        objectness_targets = []
        class_targets = []
        bbox_targets = []

        for target in targets:

            obj_tgt, cls_tgt, box_tgt = encode_targets(
                target["boxes"],
                target["labels"]
            )

            objectness_targets.append(obj_tgt)
            class_targets.append(cls_tgt)
            bbox_targets.append(box_tgt)

        objectness_targets = torch.stack(
            objectness_targets
        ).to(device)

        class_targets = torch.stack(
            class_targets
        ).to(device)

        bbox_targets = torch.stack(
            bbox_targets
        ).to(device)

        objectness_pred, class_pred, bbox_pred = model(
            images
        )

        losses = criterion(

            objectness_pred,
            class_pred,
            bbox_pred,

            objectness_targets,
            class_targets,
            bbox_targets
        )

        loss = losses["total_loss"]

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    return running_loss / len(loader)

In [ ]:
# Define one validation epoch without gradient updates.

@torch.no_grad()
def validate_one_epoch(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    running_loss = 0.0

    progress_bar = tqdm(
        loader,
        desc="Validation",
        leave=False
    )

    for images, targets in progress_bar:

        images = images.to(device)

        objectness_targets = []
        class_targets = []
        bbox_targets = []

        for target in targets:

            obj_tgt, cls_tgt, box_tgt = encode_targets(
                target["boxes"],
                target["labels"]
            )

            objectness_targets.append(obj_tgt)
            class_targets.append(cls_tgt)
            bbox_targets.append(box_tgt)

        objectness_targets = torch.stack(
            objectness_targets
        ).to(device)

        class_targets = torch.stack(
            class_targets
        ).to(device)

        bbox_targets = torch.stack(
            bbox_targets
        ).to(device)

        objectness_pred, class_pred, bbox_pred = model(
            images
        )

        losses = criterion(

            objectness_pred,
            class_pred,
            bbox_pred,

            objectness_targets,
            class_targets,
            bbox_targets
        )

        loss = losses["total_loss"]

        running_loss += loss.item()

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    return running_loss / len(loader)

In [ ]:
# Define an early-stopping helper based on validation loss.

class EarlyStopping:

    def __init__(
        self,
        patience=5,
        min_delta=0.0
    ):

        self.patience = patience
        self.min_delta = min_delta

        self.best_loss = float("inf")

        self.counter = 0

        self.early_stop = False

    def __call__(self, val_loss):

        if val_loss < self.best_loss - self.min_delta:

            self.best_loss = val_loss

            self.counter = 0

        else:

            self.counter += 1

            print(
                f"EarlyStopping Counter: "
                f"{self.counter}/{self.patience}"
            )

            if self.counter >= self.patience:

                self.early_stop = True

In [ ]:
# Train the detector, save the best checkpoint, and stop when validation loss stalls.

# ==========================================================
# Phase 9.5 - Training Configuration
# ==========================================================

NUM_EPOCHS = 200

early_stopping = EarlyStopping(
    patience=10
)

train_losses = []

val_losses = []

learning_rates = []

# ==========================================================
# Phase 9.6 - Training Loop
# ==========================================================

best_val_loss = float("inf")

for epoch in range(NUM_EPOCHS):

    print("\n" + "=" * 60)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print("=" * 60)

    # --------------------------------------------------
    # Current LR
    # --------------------------------------------------

    current_lr = optimizer.param_groups[0]["lr"]

    learning_rates.append(current_lr)

    print(f"Learning Rate: {current_lr:.6f}")

    # --------------------------------------------------
    # Train
    # --------------------------------------------------

    train_loss = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    # --------------------------------------------------
    # Validate
    # --------------------------------------------------

    val_loss = validate_one_epoch(
        model,
        val_loader,
        criterion,
        device
    )

    # --------------------------------------------------
    # Scheduler Step
    # --------------------------------------------------

    scheduler.step(val_loss)

    # --------------------------------------------------
    # Store History
    # --------------------------------------------------

    train_losses.append(train_loss)

    val_losses.append(val_loss)

    # --------------------------------------------------
    # Print
    # --------------------------------------------------

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")

    # --------------------------------------------------
    # Save Best Model
    # --------------------------------------------------

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            "best_pcb_detector.pth"
        )

        print("﷿﷿﷿﷿﷿﷿﷿﷿﷿ Best model saved.")

    # --------------------------------------------------
    # Early Stopping
    # --------------------------------------------------

    early_stopping(val_loss)

    if early_stopping.early_stop:

        print("\nEarly stopping triggered.")
        break

In [ ]:
# Plot the train and validation loss curves.

plt.figure(figsize=(8,5))

plt.plot(
    train_losses,
    label="Train Loss"
)

plt.plot(
    val_losses,
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Curves")

plt.legend()
plt.grid()

plt.show()

In [ ]:
# Plot the learning-rate schedule across epochs.

plt.figure(figsize=(8,5))

plt.plot(
    learning_rates,
    marker="o"
)

plt.xlabel("Epoch")
plt.ylabel("Learning Rate")

plt.title(
    "Learning Rate Schedule"
)

plt.grid()

plt.show()

## Phase 10: Load the Best Model and Visualize Predictions


In [ ]:
# Load the best saved detector checkpoint for inference.

# ==========================================================
# Phase 10.1 - Load Best Model
# ==========================================================

model = PCBDetector(
    num_classes=len(CLASSES)
).to(device)

model.load_state_dict(
    torch.load(
        "/kaggle/input/models/alyahmed1010/bestpcb2/pytorch/default/1/best_pcb_detector (1).pth",
        map_location=device
    )
)

model.eval()

print("Best model loaded.")

In [ ]:
# Decode grid predictions into image-space boxes, labels, and confidence scores.

# ==========================================================
# Phase 10.2 - Decode Predictions
# ==========================================================


def decode_predictions(
    objectness,
    class_logits,
    bbox_preds,
    conf_threshold=0.50,
    image_size=1024,
    grid_size=64
):

    boxes = []
    labels = []
    scores = []

    cell_size = image_size / grid_size

    objectness = torch.sigmoid(
        objectness
    )

    class_probs = torch.softmax(
        class_logits,
        dim=0
    )

    ys, xs = torch.where(
        objectness > conf_threshold
    )

    for y, x in zip(ys, xs):

        score = objectness[y, x]

        label = torch.argmax(
            class_probs[:, y, x]
        )

        # ------------------------------------------
        # Predicted Offsets
        # ------------------------------------------

        tx = bbox_preds[0, y, x]
        ty = bbox_preds[1, y, x]

        # ------------------------------------------
        # Recover Global Center
        # ------------------------------------------

        cx = (
            (x + tx)
            * cell_size
        )

        cy = (
            (y + ty)
            * cell_size
        )

        # ------------------------------------------
        # Width / Height
        # ------------------------------------------

        w = (
            bbox_preds[2, y, x]
            * image_size
        )

        h = (
            bbox_preds[3, y, x]
            * image_size
        )

        xmin = cx - (w / 2)
        ymin = cy - (h / 2)

        xmax = cx + (w / 2)
        ymax = cy + (h / 2)

        boxes.append([
            xmin.item(),
            ymin.item(),
            xmax.item(),
            ymax.item()
        ])

        labels.append(
            label.item()
        )

        scores.append(
            score.item()
        )

    return (
        boxes,
        labels,
        scores
    )

In [ ]:
# Run inference on one test image.

# ==========================================================
# Phase 10.3 - Single Image Inference
# ==========================================================

sample_idx = 0

image, target = test_dataset[
    sample_idx
]

x = image.unsqueeze(0).to(device)

with torch.no_grad():

    objectness, class_logits, bbox_preds = model(x)

print("Before squeeze:")
print(objectness.shape)
print(class_logits.shape)
print(bbox_preds.shape)

objectness = objectness.squeeze(0).squeeze(0)
class_logits = class_logits.squeeze(0)
bbox_preds = bbox_preds.squeeze(0)

print("\nAfter squeeze:")
print(objectness.shape)
print(class_logits.shape)
print(bbox_preds.shape)

In [ ]:
# Inspect objectness probabilities for the sample prediction.

# ==========================================================
# Phase 10.4 - Objectness Diagnostics
# ==========================================================

obj_prob = torch.sigmoid(
    objectness
)

print(
    "Min:",
    obj_prob.min().item()
)

print(
    "Max:",
    obj_prob.max().item()
)

print(
    "Mean:",
    obj_prob.mean().item()
)

In [ ]:
# Sweep confidence thresholds to compare predicted counts with ground truth.

# ==========================================================
# Phase 10.5 - Threshold Sweep
# ==========================================================

num_gt = len(target["boxes"])

print(f"Ground Truth Boxes = {num_gt}\n")

for threshold in [

    0.01,
    0.02,
    0.05,
    0.10,
    0.20,
    0.30,
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
    0.95,
    0.96,
    0.97,
    0.98,
    0.99,
    0.995,
    0.997,
    0.999

]:

    pred_boxes, pred_labels, pred_scores = (
        decode_predictions(
            objectness,
            class_logits,
            bbox_preds,
            conf_threshold=threshold
        )
    )

    print(
        f"Threshold={threshold:.2f}"
        f" | GT={num_gt}"
        f" | Pred={len(pred_boxes)}"
    )

In [ ]:
# Choose the confidence threshold used for later visualization and evaluation.

CONF_THRESHOLD = 0.97

In [ ]:
# Decode predictions for the current sample using the selected threshold.

# ==========================================================
# Phase 10.6 - Decode Predictions
# ==========================================================

pred_boxes, pred_labels, pred_scores = (
    decode_predictions(
        objectness,
        class_logits,
        bbox_preds,
        conf_threshold=CONF_THRESHOLD
    )
)

print(
    "Predictions:",
    len(pred_boxes)
)

if len(pred_scores) > 0:

    print(
        "Max Score:",
        max(pred_scores)
    )

In [ ]:
# Visualize predictions and ground truth on random test images.

# ==========================================================
# Phase 10.7 - Visualize 20 Random Test Images
# ==========================================================


NUM_SAMPLES = 20

indices = random.sample(
    range(len(test_dataset)),
    NUM_SAMPLES
)

mean = np.array(
    [0.485, 0.456, 0.406]
)

std = np.array(
    [0.229, 0.224, 0.225]
)

fig, axes = plt.subplots(
    NUM_SAMPLES,
    1,
    figsize=(12, NUM_SAMPLES * 5)
)

if NUM_SAMPLES == 1:
    axes = [axes]

for ax, idx in zip(axes, indices):

    image, target = test_dataset[idx]

    x = image.unsqueeze(0).to(device)

    with torch.no_grad():

        objectness, class_logits, bbox_preds = model(x)

    objectness = objectness.squeeze(0).squeeze(0)
    class_logits = class_logits.squeeze(0)
    bbox_preds = bbox_preds.squeeze(0)

    pred_boxes, pred_labels, pred_scores = (
        decode_predictions(
            objectness,
            class_logits,
            bbox_preds,
            conf_threshold=CONF_THRESHOLD
        )
    )

    # ----------------------------------------
    # De-normalize image
    # ----------------------------------------

    img = image.permute(
        1,
        2,
        0
    ).cpu().numpy()

    img = img * std + mean

    img = np.clip(
        img,
        0,
        1
    )

    ax.imshow(img)

    # ----------------------------------------
    # Ground Truth (Green)
    # ----------------------------------------

    for box in target["boxes"]:

        xmin, ymin, xmax, ymax = (
            box.numpy()
        )

        rect = patches.Rectangle(
            (xmin, ymin),
            xmax - xmin,
            ymax - ymin,
            linewidth=2,
            edgecolor="lime",
            fill=False
        )

        ax.add_patch(rect)

    # ----------------------------------------
    # Predictions (Red)
    # ----------------------------------------

    for box, label, score in zip(
        pred_boxes,
        pred_labels,
        pred_scores
    ):

        xmin, ymin, xmax, ymax = box

        rect = patches.Rectangle(
            (xmin, ymin),
            xmax - xmin,
            ymax - ymin,
            linewidth=2,
            edgecolor="red",
            fill=False
        )

        ax.add_patch(rect)

        ax.text(
            xmin,
            ymin - 5,
            f"{ID_TO_CLASS[label]} {score:.2f}",
            fontsize=7,
            color="red",
            bbox=dict(
                facecolor="white",
                alpha=0.7
            )
        )

    ax.set_title(
        f"Sample {idx} | GT={len(target['boxes'])} | Pred={len(pred_boxes)}"
    )

    ax.axis("off")

plt.tight_layout()

plt.show()

## Phase 11: Evaluate Detection and Class-Level Performance


In [ ]:
# Compute intersection-over-union between two bounding boxes.

# ==========================================================
# Phase 11.1 - IoU
# ==========================================================

def compute_iou(box1, box2):

    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])

    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection = max(
        0,
        x2 - x1
    ) * max(
        0,
        y2 - y1
    )

    area1 = (
        (box1[2] - box1[0])
        *
        (box1[3] - box1[1])
    )

    area2 = (
        (box2[2] - box2[0])
        *
        (box2[3] - box2[1])
    )

    union = (
        area1 +
        area2 -
        intersection
    )

    if union == 0:
        return 0.0

    return intersection / union

In [ ]:
# Match predicted boxes to ground-truth boxes using an IoU threshold.

# ==========================================================
# Phase 11.2 - Matching
# ==========================================================

def match_boxes(

    pred_boxes,
    gt_boxes,

    iou_threshold=0.5
):

    matched_gt = set()

    tp = 0
    fp = 0

    ious = []

    for pred_box in pred_boxes:

        best_iou = 0
        best_gt = -1

        for gt_idx, gt_box in enumerate(gt_boxes):

            if gt_idx in matched_gt:
                continue

            iou = compute_iou(
                pred_box,
                gt_box
            )

            if iou > best_iou:

                best_iou = iou
                best_gt = gt_idx

        if best_iou >= iou_threshold:

            tp += 1

            matched_gt.add(
                best_gt
            )

            ious.append(
                best_iou
            )

        else:

            fp += 1

    fn = (
        len(gt_boxes)
        - len(matched_gt)
    )

    return tp, fp, fn, ious

In [ ]:
# Evaluate detection performance across the test dataset.

# ==========================================================
# Phase 11.3 - Test Evaluation
# ==========================================================


total_tp = 0
total_fp = 0
total_fn = 0

all_ious = []

for idx in tqdm(
    range(len(test_dataset))
):

    image, target = test_dataset[idx]

    x = image.unsqueeze(0).to(device)

    with torch.no_grad():

        objectness, class_logits, bbox_preds = model(x)

    objectness = objectness.squeeze(0).squeeze(0)

    class_logits = class_logits.squeeze(0)

    bbox_preds = bbox_preds.squeeze(0)

    pred_boxes, pred_labels, pred_scores = (
        decode_predictions(
            objectness,
            class_logits,
            bbox_preds,
            conf_threshold=0.97
        )
    )

    gt_boxes = [
        box.tolist()
        for box in target["boxes"]
    ]

    tp, fp, fn, ious = match_boxes(
        pred_boxes,
        gt_boxes,
        iou_threshold=0.1
    )

    total_tp += tp
    total_fp += fp
    total_fn += fn

    all_ious.extend(
        ious
    )

In [ ]:
# Calculate and print precision, recall, F1 score, and average IoU.

# ==========================================================
# Phase 11.4 - Metrics
# ==========================================================

precision = (
    total_tp
    /
    (total_tp + total_fp + 1e-8)
)

recall = (
    total_tp
    /
    (total_tp + total_fn + 1e-8)
)

f1 = (
    2
    *
    precision
    *
    recall
    /
    (precision + recall + 1e-8)
)


print("\n")
print("=" * 60)
print("TEST RESULTS")
print("=" * 60)

print(
    f"Precision : {precision:.4f}"
)

print(
    f"Recall    : {recall:.4f}"
)

print(
    f"F1 Score  : {f1:.4f}"
)


In [ ]:
# Compare dominant predicted classes with ground-truth classes per test image.

correct_class = 0

results = []

for idx in tqdm(range(len(test_dataset))):

    image, target = test_dataset[idx]

    gt_class = test_records[idx]["class_name"]

    x = image.unsqueeze(0).to(device)

    with torch.no_grad():

        objectness, class_logits, bbox_preds = model(x)

    objectness = objectness.squeeze(0).squeeze(0)
    class_logits = class_logits.squeeze(0)
    bbox_preds = bbox_preds.squeeze(0)

    pred_boxes, pred_labels, pred_scores = (
        decode_predictions(
            objectness,
            class_logits,
            bbox_preds,
            conf_threshold=0.97
        )
    )

    if len(pred_labels) == 0:

        pred_class = "None"

    else:

        most_common = Counter(
            pred_labels
        ).most_common(1)[0][0]

        pred_class = ID_TO_CLASS[
            most_common
        ]

    if pred_class == gt_class:

        correct_class += 1

    results.append(
        (
            gt_class,
            pred_class,
            len(target["boxes"]),
            len(pred_boxes)
        )
    )

print(
    "Image Classification Accuracy:",
    correct_class / len(test_dataset)
)

In [ ]:
# Calculate the average absolute error in predicted defect counts.

count_errors = []

for gt_class, pred_class, gt_count, pred_count in results:

    count_errors.append(
        abs(
            gt_count - pred_count
        )
    )

print(
    "Mean Count Error:",
    sum(count_errors)
    /
    len(count_errors)
)

In [ ]:
# Print a random sample of class and count prediction results.

for sample in random.sample(results, 20):

    gt_class, pred_class, gt_count, pred_count = sample

    print(
        f"GT={gt_class:<18}"
        f" Pred={pred_class:<18}"
        f" GT_Count={gt_count:<2}"
        f" Pred_Count={pred_count:<2}"
    )

In [ ]:
# Estimate class prediction accuracy over all predicted boxes.

correct_preds = 0
total_preds = 0

for idx in range(len(test_dataset)):

    image, target = test_dataset[idx]

    gt_label = target["labels"][0].item()

    x = image.unsqueeze(0).to(device)

    with torch.no_grad():

        objectness, class_logits, bbox_preds = model(x)

    objectness = objectness.squeeze(0).squeeze(0)
    class_logits = class_logits.squeeze(0)
    bbox_preds = bbox_preds.squeeze(0)

    pred_boxes, pred_labels, pred_scores = (
        decode_predictions(
            objectness,
            class_logits,
            bbox_preds,
            conf_threshold=0.97
        )
    )

    total_preds += len(pred_labels)

    for pred_label in pred_labels:

        if pred_label == gt_label:

            correct_preds += 1

print(
    "Per-Prediction Class Accuracy:",
    correct_preds / max(total_preds, 1)
)